# Lab 03 — Part 3: Polymer in Poiseuille Flow (standalone)

This notebook is **self-contained**: it installs ESPResSo from source, runs a
quick LBM sanity check, then simulates a polymer in Poiseuille flow and
compares the effective diffusion coefficient to the Taylor–Aris prediction.

**Runtime**: ~30–40 min on Colab CPU (3 force densities).


## 0. Installation

In [ ]:
# Install system and Python dependencies
!apt-get update -qq
!apt-get install -y cmake g++ libfftw3-dev libhdf5-dev libboost-all-dev \
    openmpi-bin libopenmpi-dev
!pip install -q numpy scipy matplotlib h5py


In [ ]:
# Clone ESPResSo 4.2
!git clone --recursive --single-branch -b 4.2 \
    https://github.com/espressomd/espresso.git /content/espresso


In [ ]:
# Build ESPResSo (takes ~15 min on Colab CPU)

%mkdir -p /content/espresso/build
%cd /content/espresso/build
!cmake .. -DPYTHON=ON -DENABLE_PYTHON=ON \
         -DCMAKE_INSTALL_PREFIX=/content/espresso/install \
         -DWITH_CUDA=OFF
!make -j2
!make install

In [ ]:
# 1.2.1 Importing necessary packages

# Ensure espressomd is on the path regardless of Python version
import sys, glob
matches = glob.glob("/content/espresso/install/**/espressomd", recursive=True)
if matches:
    sys.path.insert(0, matches[0].rsplit("/espressomd",1)[0])
else: 
    raise RuntimeError("espressomd not found — did the build complete without errors?")

import logging
import threading
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import scipy.optimize
import espressomd
import espressomd.analyze
import espressomd.accumulators
import espressomd.observables
import espressomd.polymer
import espressomd.io.writer.vtf as vtf
import espressomd.visualization as viz
import espressomd.lb
import espressomd.lbboundaries
import espressomd.shapes
import espressomd.constraints
import espressomd.interactions

## 1. LBM sanity check — Poiseuille profile (no polymer)

Before adding the polymer we verify that the LBM fluid with bounce-back walls
produces the correct parabolic velocity profile.  
`kT=0` removes thermal noise so the profile is clean.

**Runs in ~1–2 min.**


In [ ]:
# Parameters
BOX   = 16       # box size (same in all directions)
AGRID = 1.0      # LB grid spacing
VISC  = 5.0      # kinematic viscosity
DENS  = 1.0
DT    = 0.01
F_TEST = 0.003   # body-force density for the sanity check

# Analytical v_max = F * H^2 / (8 eta),  H = BOX - 2*AGRID (channel width)
eta   = VISC * DENS
H     = BOX - 2 * AGRID
v_max_ana = F_TEST * H**2 / (8 * eta)
print(f"Analytical v_max = {v_max_ana:.5f}")


In [ ]:
# Build system + LBM fluid with bounce-back walls 
system_check = espressomd.System(box_l=[BOX]*3)
system_check.periodicity     = [True, True, True]
system_check.time_step       = DT
system_check.cell_system.skin = 0.4

# Force in Y, walls in X (same convention as Part 2 of the main lab)
lbf_check = espressomd.lb.LBFluid(
    kT=0, agrid=AGRID, dens=DENS, visc=VISC, tau=DT,
    ext_force_density=[0.0, F_TEST, 0.0])
system_check.actors.add(lbf_check)

system_check.lbboundaries.add(espressomd.lbboundaries.LBBoundary(
    shape=espressomd.shapes.Wall(normal=[1, 0, 0], dist=AGRID)))
system_check.lbboundaries.add(espressomd.lbboundaries.LBBoundary(
    shape=espressomd.shapes.Wall(normal=[-1, 0, 0], dist=-(BOX - AGRID))))

system_check.integrator.run(5000)
print("Integration done.")


In [ ]:
# Read velocity profile and plot
x_cen  = np.arange(BOX) + 0.5
v_sim  = np.array([lbf_check[j, 0, 0].velocity[1] for j in range(BOX)])

x0, x1 = AGRID, BOX - AGRID
v_ana  = np.where((x_cen > x0) & (x_cen < x1),
                  F_TEST / (2*eta) * (x_cen - x0) * (x1 - x_cen), 0.0)

plt.figure(figsize=(5, 4))
plt.plot(v_sim, x_cen, 'o', ms=5, label='LBM')
plt.plot(v_ana, x_cen, '--',      label='Analytical')
plt.xlabel(r"$v_y$  (flow direction)")
plt.ylabel(r"$x$  (wall-normal)")
plt.title("Poiseuille profile — LBM sanity check")
plt.legend(); plt.tight_layout(); plt.show()

print(f"v_max  simulation : {v_sim.max():.5f}")
print(f"v_max  analytical : {v_max_ana:.5f}")
print(f"relative error    : {abs(v_sim.max()-v_max_ana)/v_max_ana*100:.2f}%")


## 2. Polymer in Poiseuille flow

We add a short polymer chain (Rouse, 10 beads) to the Poiseuille flow and
measure the centre-of-mass MSD in all three directions.

**Taylor–Aris prediction (slit channel):**

$$\frac{D_{\parallel}}{D_0} = 1 + \frac{\mathrm{Pe}^2}{210}, \qquad
\mathrm{Pe} = \frac{v_{\max} R_g}{D_0}$$


In [ ]:
# Simulation parameters
N_MON    = 10       # monomers
BOX_L    = 16.0     # cubic box
LB_AGRID = 1.0
LB_VISC  = 5.0
LB_DENS  = 1.0
GAMMA    = 5.0
KT       = 1.0
TIME_STEP = 0.01

LOOPS    = 400      # production loops
STEPS    = 200      # integrator steps per loop  → 80 k steps total
TAU_MAX  = LOOPS * STEPS * TIME_STEP * 0.5

FORCE_DENSITIES = [0.0, 0.003, 0.010]  # Pe ≈ 0, moderate, high

# Derived channel geometry
ETA  = LB_VISC * LB_DENS
X0   = LB_AGRID
X1   = BOX_L - LB_AGRID
H_CH = X1 - X0      # effective channel width

print(f"Channel width H = {H_CH:.1f}")
print(f"tau_max = {TAU_MAX:.1f} time units")


In [ ]:
# Simulation function
# ESPResSo only allows ONE System per Python session.  We reuse the one
# already created in Section 1 (system_check) rather than creating a new one.

_fene = None

def run_poiseuille(f_drive, n_monomers=N_MON, seed=42):
    global _fene

    # Reuse existing System, reset state
    system = system_check
    system.part.clear()
    system.thermostat.turn_off()
    system.auto_update_accumulators.clear()
    system.constraints.clear()
    system.lbboundaries.clear()
    system.actors.clear()

    # Interactions
    system.non_bonded_inter[0, 0].lennard_jones.set_params(
        epsilon=1.0, sigma=1.0, cutoff=2.0**(1/6), shift="auto")

    if _fene is None:
        _fene = espressomd.interactions.FeneBond(k=7, r_0=1, d_r_max=2)
        system.bonded_inter.add(_fene)

    # Polymer
    start_pos = np.array([[BOX_L/2, BOX_L/2, BOX_L/2]])
    positions = espressomd.polymer.linear_polymer_positions(
        n_polymers=1, beads_per_chain=n_monomers,
        bond_length=1.0, seed=seed, min_distance=0.9,
        start_positions=start_pos)

    pids = []
    for i, pos in enumerate(positions[0]):
        p = system.part.add(pos=pos, type=0)
        if i > 0:
            p.add_bond((_fene, pids[-1]))
        pids.append(p.id)

    # LBM fluid: force in Y, walls in X
    lbf = espressomd.lb.LBFluid(
        kT=KT, seed=seed, agrid=LB_AGRID,
        dens=LB_DENS, visc=LB_VISC, tau=TIME_STEP,
        ext_force_density=[0.0, f_drive, 0.0])
    system.actors.add(lbf)

    system.lbboundaries.add(espressomd.lbboundaries.LBBoundary(
        shape=espressomd.shapes.Wall(normal=[1, 0, 0], dist=LB_AGRID)))
    system.lbboundaries.add(espressomd.lbboundaries.LBBoundary(
        shape=espressomd.shapes.Wall(normal=[-1, 0, 0],
                                     dist=-(BOX_L - LB_AGRID))))

    system.constraints.add(espressomd.constraints.ShapeBasedConstraint(
        shape=espressomd.shapes.Wall(normal=[1, 0, 0], dist=1.5),
        particle_type=0, penetrable=False))
    system.constraints.add(espressomd.constraints.ShapeBasedConstraint(
        shape=espressomd.shapes.Wall(normal=[-1, 0, 0],
                                     dist=-(BOX_L - 1.5)),
        particle_type=0, penetrable=False))

    # Equilibration
    system.integrator.set_steepest_descent(
        f_max=0, gamma=0.1, max_displacement=0.05)
    system.integrator.run(500)

    system.thermostat.set_lb(LB_fluid=lbf, gamma=GAMMA, seed=seed)
    system.integrator.set_vv()
    system.integrator.run(5000)

    # MSD correlator 
    com_pos = espressomd.observables.ComPosition(ids=pids)
    msd_cor = espressomd.accumulators.Correlator(
        obs1=com_pos, tau_lin=16, tau_max=TAU_MAX, delta_N=5,
        corr_operation="square_distance_componentwise",
        compress1="discard1")
    system.auto_update_accumulators.add(msd_cor)

    # Production — time-average the velocity profile
    # A single LBM snapshot at kT=1 is dominated by thermal noise (v_thermal~1
    # vs v_flow~0.015).  Averaging over all production steps recovers the mean.
    v_accum = np.zeros(int(BOX_L))
    n_accum = 0
    for step in range(LOOPS):
        system.integrator.run(STEPS)
        if step % 5 == 0:   # sample every 5 loops
            v_accum += np.array([lbf[j, 0, 0].velocity[1]
                                  for j in range(int(BOX_L))])
            n_accum += 1
        if step % 100 == 0:
            print(f'  loop {step}/{LOOPS}', flush=True)

    msd_cor.finalize()
    msd  = np.array(msd_cor.result())
    lag  = np.array(msd_cor.lag_times())
    v_prof = v_accum / n_accum   # time-averaged profile

    return msd, lag, v_prof

print('Function defined.')


In [ ]:
# Verify walls survive the polymer thermostat
# At kT=1 the thermal velocity (~1.0) swamps the flow signal (~0.015) even
# after time-averaging.  We use a large force (f=1.0, v_max~2.5) so the
# parabola is clearly visible above the noise.
_, _, v_chk = run_poiseuille(f_drive=1.0)

x_cen  = np.arange(int(BOX_L)) + 0.5
f_big  = 1.0
v_ana  = np.where((x_cen > X0) & (x_cen < X1),
                  f_big / (2*ETA) * (x_cen - X0) * (X1 - x_cen), 0.0)

plt.figure(figsize=(5, 4))
plt.plot(v_chk, x_cen, 'o', ms=5, label='LBM + polymer thermostat')
plt.plot(v_ana, x_cen, '--',      label='Analytical')
plt.xlabel(r'$v_y$  (flow direction)')
plt.ylabel(r'$x$  (wall-normal)')
plt.title('Wall check at large force (f=1.0, high SNR)')
plt.legend(); plt.tight_layout(); plt.show()

v_max_ana = f_big * (X1 - X0)**2 / (8*ETA)
print(f'v_max  sim={v_chk.max():.4f}  analytical={v_max_ana:.4f}')
print('Walls OK if sim ~ analytical')


In [ ]:
# Pe scan 
results = {}
for f_drive in FORCE_DENSITIES:
    v_max = f_drive * H_CH**2 / (8*ETA)
    print(f"\nf_drive={f_drive}  v_max={v_max:.5f}")
    msd, lag, _ = run_poiseuille(f_drive=f_drive)
    results[f_drive] = {"msd": msd, "lag": lag, "v_max": v_max}

print("\nDone.")


In [ ]:
# Extract D from MSD linear regime
# Raw MSD_y = 2·D_eff·τ + v_mean²·τ²  (drift term from flow carries polymer in Y)
# v_mean = 2/3·v_max  (spatial average of parabolic profile)
# Subtract the drift term before fitting so we recover D_eff alone.

def fit_D(lag, msd_1d, frac_lo=0.35, frac_hi=0.65):
    """Fit MSD = 2 D t in the central fraction of the lag-time window."""
    n = len(lag)
    lo, hi = int(n * frac_lo), int(n * frac_hi)
    if hi <= lo + 2:
        return np.nan
    slope, _ = np.polyfit(lag[lo:hi], msd_1d[lo:hi], 1)
    return slope / 2.0

Pe_list, D_par_list, D_perp_list = [], [], []

# D0 from f=0 run (no drift, no correction needed)
f0   = FORCE_DENSITIES[0]
lag0 = results[f0]['lag']
msd0 = results[f0]['msd']
D0   = fit_D(lag0, msd0[:, 1])   # Y at f=0: free diffusion, no drift
D0_x = fit_D(lag0, msd0[:, 0])
D0_z = fit_D(lag0, msd0[:, 2])
Rg_est = 1.0  # replace with Rg from Part 1 for accurate Pe scale

print(f'D0 (Y, f=0, free) = {D0:.4f}  <-- baseline')
print(f'D0_x (confined)   = {D0_x:.4f}')
print(f'D0_z (vorticity)  = {D0_z:.4f}')
print()

for f_drive, res in results.items():
    lag   = res['lag']
    msd   = res['msd']
    v_max = res['v_max']

    # Drift correction for flow direction (Y)
    v_mean = (2/3) * v_max          # spatial mean of parabolic profile
    msd_y_corr = msd[:, 1] - v_mean**2 * lag**2

    D_y = fit_D(lag, msd_y_corr)    # D_parallel, drift-corrected
    D_x = fit_D(lag, msd[:, 0])     # D_x, confined
    D_z = fit_D(lag, msd[:, 2])     # D_z, vorticity
    D_t = (D_x + D_z) / 2           # mean transverse = D_perp

    Pe  = v_max * Rg_est / D0 if D0 > 0 else 0
    Pe_list.append(Pe)
    D_par_list.append(D_y)
    D_perp_list.append(D_t)
    print(f'f={f_drive:.4f}  Pe={Pe:.2f}  D_∥={D_y:.4f}  '
          f'D_⊥={D_t:.4f}  D_∥/D0={D_y/D0:.3f}  '
          f'(drift subtracted: v_mean={v_mean:.5f})')


In [ ]:
# Taylor–Aris plot
Pe_arr     = np.array(Pe_list)
D_par_arr  = np.array(D_par_list)
D_perp_arr = np.array(D_perp_list)

Pe_theory  = np.linspace(0, max(Pe_arr) * 1.1, 100)
D_theory   = 1 + Pe_theory**2 / 210      # Taylor–Aris (slit, point particle)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))

ax1.plot(Pe_theory, D_theory, 'k--', label='Taylor–Aris')
ax1.plot(Pe_arr, D_par_arr / D0, 'o', ms=8, label=r'$D_\parallel/D_0$ sim')
ax1.plot(Pe_arr, D_perp_arr / D0, 's', ms=8, label=r'$D_\perp/D_0$ sim')
ax1.set_xlabel("Pe")
ax1.set_ylabel(r"$D/D_0$")
ax1.set_title("Effective diffusion vs Péclet number")
ax1.legend()

ax2.plot(Pe_arr**2, (D_par_arr - D0) / D0, 'o', ms=8)
Pe2_theory = np.linspace(0, max(Pe_arr**2) * 1.1, 100)
ax2.plot(Pe2_theory, Pe2_theory / 210, 'k--', label='slope = 1/210')
ax2.set_xlabel(r"$\mathrm{Pe}^2$")
ax2.set_ylabel(r"$(D_\parallel - D_0)/D_0$")
ax2.set_title("Taylor–Aris linearity check")
ax2.legend()

plt.tight_layout()
plt.savefig("taylor_aris.png", dpi=120)
plt.show()
print("Plot saved as taylor_aris.png")
